# Forecasting Daily Sales with XGBoost — A Walkthrough

This notebook is a worked example of a **tabular time-series forecasting** workflow built around the
[Rohlik Orders Forecasting Challenge](https://www.kaggle.com/competitions/rohlik-sales-forecasting-challenge-v2)
(an online grocery delivery dataset). We forecast the number of units sold for each `(product, warehouse, day)`
combination, given history of sales, prices, discounts, and a calendar of holidays.

This is the **XGBoost twin** of a sibling LightGBM notebook. The data prep and feature engineering are
deliberately identical; only the model layer changes. This is itself a useful pattern: when you build
a single feature pipeline and slot in different boosters, the **out-of-fold (OOF) predictions become
diversified base learners** for an ensemble. Even when XGBoost and LightGBM score similarly alone,
their average often outperforms either one — they make different mistakes.

## XGBoost in one paragraph

[XGBoost](https://xgboost.readthedocs.io/) is a gradient-boosted decision tree library. It builds an
ensemble of regression trees one at a time, each new tree fit to the residuals (gradient/Hessian) of
the current ensemble. Compared to LightGBM, XGBoost grows trees **level-wise** by default (LightGBM
grows **leaf-wise**), tends to be slightly slower per round but more conservative, and historically
had stronger GPU support. For tabular regression the two are very close in practice.

## What's the same as the LightGBM notebook

- Feature engineering (`fe_date`, `fe_other`, `fe_combined`)
- Calendar / holiday processing
- Target-derived features with the `.shift(1).ewm()` lag trick
- `RepeatedKFold` cross-validation strategy
- The square-root power transform on the target
- The 5-fold-bagging-style averaging of test predictions

## What's different

- **Imports**: `XGBRegressor` from `xgboost` instead of `LGBMRegressor` from `lightgbm`.
- **Categorical handling**: XGBoost requires `enable_categorical=True` (opt-in) and treats
  `pd.Categorical` columns directly. There is **no** explicit dtype-cleanup block in this version
  (the LightGBM notebook had one) — XGBoost's checking is more permissive but you should still
  align category levels between train and test.
- **Early stopping** is a **constructor** parameter (`early_stopping_rounds=es`) in XGBoost, not
  a `fit()` callback as in LightGBM.
- **GPU**: `device='cuda'` directly (vs LightGBM's `device_type='gpu'` and a special build).
- **Objective naming**: `'reg:squarederror'` (XGBoost) vs `'regression'` (LightGBM).
- **Verbosity**: `verbosity=0` (XGBoost) vs `verbose=-1` (LightGBM); the `verbose` arg in
  `fit()` controls evaluation logging frequency in XGBoost.


## 1. Imports

Same numerical stack as the LightGBM notebook, swapping in XGBoost. We import `DMatrix` here even
though the high-level `XGBRegressor` API doesn't strictly need it; `DMatrix` is XGBoost's optimized internal data structure (analogous
to LightGBM's `Dataset`); the sklearn-style wrapper builds them under the hood.


In [ ]:
import numpy as np
import pandas as pd
from copy import deepcopy
from sklearn.metrics import mean_absolute_error      # Competition metric is weighted MAE.
from sklearn.model_selection import RepeatedKFold    # K-fold CV; see caveat about time-series below.
from xgboost import XGBRegressor, DMatrix            # DMatrix is XGBoost's internal optimized data struct.


## 2. Load the inventory (product metadata) table


In [ ]:
inventory = pd.read_csv('inventory.csv').drop(['warehouse','product_unique_id'],axis=1)
inventory.head()


## 3. Feature engineering functions

See lgbm notebook


In [ ]:
def fe_date(df):
    # Pure date features: year, day-of-week, days since an arbitrary anchor (linear time trend),
    # and (sin, cos) of day-of-year for cyclical seasonality.
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.dayofweek
    df['days_since_2020'] = (df['date'] - pd.to_datetime('2020-01-01')).dt.days.astype('int')  # monotonic time trend
    df['day_of_year'] = df['date'].dt.dayofyear
    df['cos_day'] = np.cos(df['day_of_year']*2*np.pi/365)  # Dec 31 ~ Jan 1 in (cos, sin) space.
    df['sin_day'] = np.sin(df['day_of_year']*2*np.pi/365)

def fe_other(df):
    # The dataset records up to 7 different "discount types" per row.
    discount_cols = ['type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount']
    df[discount_cols] = df[discount_cols].clip(0)  # Replace any negative entries with 0 (data hygiene).
    # max_discount = the strongest promotion across types 0–5 on this row. (type_6 is excluded here on purpose.)
    df['max_discount'] = df[['type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount']].max(axis=1)

    # Given that we're using XGBoost, which is in theory invariant to monotonic transformations of features, this transformation in isolation doesn't really do anything. I mainly did it because it made the shap plot look more linear. However, I think it did make further feature engineering that used price more effective.
    df['sell_price_main'] = np.log(df['sell_price_main'])

    # Names look like "<family>_<variant>_<size>". Take everything before the first "_" as a coarse family key.
    df['common_name'] = df['name'].apply(lambda x: x[:x.find('_')])
    # Same-day group aggregations. transform() broadcasts the group result back to each row.
    df['CN_total_products'] = df.groupby(['date','warehouse','common_name'])['unique_id'].transform('nunique')   # how many siblings are on shelves today
    df['CN_discount_avg']   = df.groupby(['date','warehouse','common_name'])['max_discount'].transform('mean')   # avg promo intensity in the family today
    df['CN_WH'] = df['common_name'] + '_' + df['warehouse']                                                       # categorical interaction: family × warehouse
    df['name_num_warehouses'] = df.groupby(['date','name'])['unique_id'].transform('nunique')                    # geographic breadth of this exact SKU today

def fe_combined(df):
    # Rolling 28-day count of days this product appeared. closed='left' = window is [t-28, t-1], excludes today.
    # This is "how long has this product been around / how regularly does it appear".
    df['num_sales_days_28D'] = pd.MultiIndex.from_frame(df[['unique_id','date']]).map(df.sort_values('date').groupby('unique_id').rolling(
        window='28D', on='date', closed='left')['date'].count().fillna(0))

    # This 'price_detrended' feature was one I found pretty late into the game, but I think it helped out a lot. I was trying to make a feature that captured whether an item was cheap or expensive relative to its usual price, which is what 'price_scaled' represents. What I found was that the prices of things generally increase over time. So I removed that time-based trend to construct price_detrended, and that proved very effective.
    mean_prices = df.groupby(df['unique_id'])['sell_price_main'].mean()
    std_prices  = df.groupby(df['unique_id'])['sell_price_main'].std()
    # Per-product z-score of (log) price. np.where guards against std==0 (constant-price products) → produce 0.
    df['price_scaled'] = np.where(df['unique_id'].map(std_prices) == 0, 0,
                                  (df['sell_price_main'] - df['unique_id'].map(mean_prices))/df['unique_id'].map(std_prices))
    # Subtract the daily-warehouse mean of price_scaled to remove the global "prices drift up over time" trend.
    df['price_detrended'] = df['price_scaled'] - df.groupby(['days_since_2020','warehouse'])['price_scaled'].transform('mean')
    df.drop('price_scaled',axis=1,inplace=True)  # only keep the detrended version

    # Warehouse-level demand index. Use median across products inside each (date, warehouse) for robustness,
    # then smooth with a 14-day rolling mean and a 56-day EWM. These are exogenous proxies for
    # "is the whole warehouse busy today?".
    warehouse_stats = df.groupby(['date','warehouse'])['total_orders'].median().rename('med_total_orders').reset_index().sort_values('date')
    warehouse_stats['ewmean_orders_56'] = warehouse_stats.groupby('warehouse')['med_total_orders'].transform(lambda x:x.ewm(alpha=1/56).mean())
    df['mean_orders_14d'] = pd.MultiIndex.from_frame(df[['warehouse','date']]).map(
        warehouse_stats.groupby('warehouse').rolling(on='date',window='14D')['med_total_orders'].mean())
    df['ewmean_orders_56'] = pd.MultiIndex.from_frame(df[['warehouse','date']]).map(
        warehouse_stats.set_index(['warehouse','date'])['ewmean_orders_56'])
    return df


## 4. Calendar features — proximity to holidays

See lgbm notebook


In [ ]:
calendar = pd.read_csv('calendar.csv', parse_dates=['date'])
calendar.loc[calendar['holiday_name'].isna(), 'holiday'] = 0  # V3: fix mislabeled holidays with no name

# Mark today's date in two helper columns ONLY on actual holiday rows; everything else NaN.
calendar['last_holiday_date'] = calendar['date']
calendar['next_holiday_date'] = calendar['date']
calendar.loc[calendar['holiday'] == 0, ['last_holiday_date','next_holiday_date']] = np.nan

# Within each warehouse, propagate the last/next holiday dates forward/backward in time.
calendar['last_holiday_date'] = calendar.sort_values('date').groupby('warehouse')['last_holiday_date'].ffill()
calendar['next_holiday_date'] = calendar.sort_values('date').groupby('warehouse')['next_holiday_date'].bfill()

# Compute integer day distances and derive the two binary flags we'll actually keep.
calendar['days_since_last_holiday'] = ((calendar['date'] - calendar['last_holiday_date']).dt.days)
calendar['days_to_next_holiday']    = ((calendar['next_holiday_date'] - calendar['date']).dt.days)
calendar['day_before_holiday'] = calendar['days_to_next_holiday'] == 1
calendar['day_after_holiday']  = calendar['days_since_last_holiday'] == 1

# Drop scaffolding + columns the author found uninformative in this competition.
calendar.drop(['last_holiday_date','next_holiday_date'],axis=1,inplace=True)
calendar.drop(['days_since_last_holiday','days_to_next_holiday'],axis=1,inplace=True)
calendar.drop(['shops_closed','winter_school_holidays','school_holidays','holiday_name'],axis=1,inplace=True)


## 5. Assemble the train/test datasets

See lgbm notebook


In [ ]:
# ---- TRAIN ----
train = pd.read_csv('sales_train.csv', parse_dates=['date'])
train['id'] = train['unique_id'].astype('str') + '_' + train['date'].astype('str')
train.set_index('id',inplace=True)
train = train[~train['sales'].isna()]   # drop rows with missing target
# Merge static product metadata; reset/set index restores original row order via .loc[train.index].
train = train.reset_index().merge(inventory, on='unique_id').set_index('id').loc[train.index]
# Merge per-day calendar features (holiday flags + before/after).
train = train.reset_index().merge(calendar, on=['date','warehouse']).set_index('id').loc[train.index]
fe_date(train)
fe_other(train)

# ---- TEST ----  (same pattern, but no target to filter on)
test = pd.read_csv('sales_test.csv', parse_dates=['date'])
test['id'] = test['unique_id'].astype('str') + '_' + test['date'].astype('str')
test.set_index('id',inplace=True)
test = test.reset_index().merge(inventory, on='unique_id').set_index('id').loc[test.index]
test = test.reset_index().merge(calendar, on=['date','warehouse']).set_index('id')
fe_date(test)
fe_other(test)

# ---- COMBINED FEATURES ----
# fe_combined needs both halves (e.g. for per-product mean price). It does NOT use `sales`, so this is leakage-safe.
all_data = pd.concat([train,test])
all_data = fe_combined(all_data)
train = all_data.loc[train.index]
test  = all_data.loc[test.index].drop(['sales','availability'],axis=1)


## 6. Sanity-check a single product's test rows

Always look at your data after a heavy merge/feature-engineering pipeline. Pick one `unique_id`,
sort by date, and eyeball: does it have the expected number of rows? Are the engineered columns
populated? Any unexpected NaNs?


In [ ]:
test.sort_values(by=['unique_id','date'], ascending=True)[test['unique_id']==1]


## 7. Split into features (X), target (y), and sample weights

Three things happen here:

1. **Split off the target.** `y_train = train['sales']`, then drop it from `X_train`.
2. **Stash `availability`.** It's a *historical-only* column (not present in test), so we can't use
   it as a feature for prediction, but we keep it aside in case we want it for diagnostics or
   sample weighting later.
3. **Per-product sample weights.** The competition metric is **weighted MAE** — different products
   carry different weights. We pull weights from `test_weights.csv` and broadcast them onto every
   row of training via a `unique_id → weight` map.

> **Why weights matter:** If product A has weight 10 and product B has weight 1, a 1-unit error on
> A costs 10× as much as the same error on B. The model should "spend more attention" reducing A's
> errors. Without sample weights you'd be optimizing a metric the leaderboard isn't using.

> **Heads-up for XGBoost:** Note that the training loop below does **not** actually pass these
> weights to `xgb.fit()` — they're only used in the OOF metric computation. To use them during
> training, pass `sample_weight=X_train_weights.iloc[idx_t]` to `fit()`. This is a worthwhile
> experiment to try (the LightGBM sister notebook has the same omission).


In [ ]:
X_train = train.drop('sales',axis=1)
y_train = train['sales']
train_availability = X_train['availability']      # keep aside; we won't use it as a feature
X_train.drop('availability',inplace=True,axis=1)

# Per-product weights from the competition metric definition.
weights = pd.read_csv('test_weights.csv').set_index('unique_id')
X_train_weights = X_train['unique_id'].map(weights['weight'])  # broadcast weight to each row


## 8. Target-derived features (and the leakage trap)

See lgbm notebook

In [ ]:
cat_cols = ['unique_id'] + list(X_train.columns[X_train.dtypes == 'object'])
all_data = pd.concat([X_train, test])
add_cols = ['last_sales_ema005','CN_sales_sum','last_sales_zs']

# Here there are a few additional features engineered from historical sales data. These are done separately from the rest of my feature engineering because when I go to test model performance on a time-based holdout validation set, I need to make sure these features aren't using sales data from that validation set.

# Build a continuous calendar per product spanning [first appearance .. last test date]
# so EMAs progress on every calendar day, not just sales days.
train_cp = train.groupby('unique_id')['date'].apply(lambda s: pd.date_range(s.min(), test.date.max())).explode().reset_index()
train_cp = train_cp.merge(
    pd.concat([train[['unique_id','date','sales','warehouse',]],
               test[['unique_id','date','warehouse']]]),
    on=['unique_id','date'],how='left')
train_cp = train_cp.merge(inventory, left_on='unique_id', right_index=True)
train_cp['common_name'] = train_cp['name'].apply(lambda x: x[:x.find('_')])
train_cp.sort_values('date',inplace=True)

# IMPORTANT: .shift(1) before .ewm() — today's EMA must use only sales up to yesterday.
train_cp['last_sales_ema005'] = train_cp.groupby(['unique_id'])['sales'].transform(lambda x: x.shift(1).ewm(alpha=.005).mean()).fillna(0)
# Family-level demand on each date/warehouse: sum of per-product EMAs.
train_cp['CN_sales_sum'] = train_cp.groupby(['common_name','warehouse','date'])['last_sales_ema005'].transform('sum')

# Merge the target-derived features back into all_data.
all_data = all_data.merge(train_cp.set_index(['unique_id','date'])[[
    'last_sales_ema005','CN_sales_sum'
]], left_on=['unique_id','date'],right_index=True,how='left')

# Z-score the EMA against the family's historical mean/std (computed on the train_cp continuous frame).
sales_stats = train_cp.groupby(['common_name','warehouse'])['sales'].agg(['mean','std'])
all_data['last_sales_zs'] = (all_data['last_sales_ema005'] - pd.MultiIndex.from_frame(all_data[['common_name','warehouse']]).map(
    sales_stats['mean']))/ pd.MultiIndex.from_frame(all_data[['common_name','warehouse']]).map(sales_stats['std'])

# Cutting all data prior to 2022 seems to help. This could be due to COVID effects, and also the fact that there is little data from the Germany warehouses before 2022.
X_train = X_train[X_train['date'] >= '2022-01-01']
y_train = y_train.loc[X_train.index]
X_train_weights = X_train_weights.loc[X_train.index]

# Attach the new target-derived features to X_train and test.
X_train[add_cols] = all_data[add_cols]
test[add_cols] = all_data[add_cols]

# Final cast: every categorical column → pandas 'category' dtype, which XGBoost consumes natively
# when enable_categorical=True (set in base_params below). The .astype('str') pass first normalizes
# any mixed-type columns (e.g. unique_id is int) into strings before becoming category levels.
all_data[cat_cols] = all_data[cat_cols].astype('str').astype('category')


## 9. Hyperparameters

We define two parameter dicts: one for the model, one for the CV splitter.

### Model hyperparameters — XGBoost specifics

| Parameter | Value | Notes |
|---|---|---|
| `n_estimators` | `5000` | Upper bound; **early stopping** picks the actual count per fold. |
| `learning_rate` | `0.1` | Moderate. Smaller lr + more trees usually generalizes better. |
| `verbosity` | `0` | Library-level chatter off. (LightGBM's equivalent is `verbose=-1`.) |
| `enable_categorical` | `True` | **XGBoost-specific opt-in.** Without this, `category` dtype columns cause an error. |
| `early_stopping_rounds` | `10` | Stop after 10 rounds without validation improvement. **Set on the constructor**, unlike LightGBM (callback). |
| `objective` | `'reg:squarederror'` | XGBoost's L2 regression. (LightGBM calls this `'regression'`.) |
| `eval_metric` | `'rmse'` | Reported during training. We score MAE post-hoc. |
| `device` | `'cuda'` | GPU training. **Set to `'cpu'` if no GPU.** |
| `reg_lambda` | `0` | No L2 regularization on leaf values. |
| `min_child_weight` | `1` | Minimum sum of Hessians per leaf — XGBoost's analog of `min_child_samples`, but in *gradient* space. For squared error this is roughly equivalent to "minimum number of samples per leaf." |

### Why `early_stopping_rounds` is on the constructor (not `fit`)

Modern XGBoost (1.6+) moved most callback-style options into the sklearn-API constructor for
consistency. The fit call still accepts `eval_set` (where to evaluate the metric) and `verbose`
(how often to print). LightGBM kept the older callback approach
(`callbacks=[early_stopping(es), log_evaluation(...)]`).

### Why `device='cuda'` here

XGBoost ships with GPU support out of the box on CUDA-capable machines — no special build needed.
The newer `device` parameter (replaces the older `tree_method='gpu_hist'`) accepts `'cpu'`, `'cuda'`,
or `'cuda:N'` for a specific device. **If you don't have a GPU, change this to `'cpu'`** or training
will fail with a CUDA error.

### CV hyperparameters

- `n_splits=3, n_repeats=1` — three folds, no repetition. Each row gets exactly one OOF prediction.
- Larger `n_repeats` would give a more stable OOF estimate at proportional compute cost.

### Why `RepeatedKFold` here, even though this is a time-series problem?

Pragmatic, not principled. Random k-fold mixes future and past in the same fold, so the OOF score
will be **optimistically biased** — features like rolling means and EMAs leak information that a
forward-only deployment wouldn't have. For a truly trustworthy validation score, use
`TimeSeriesSplit` and recompute target-derived features within each fold.


In [ ]:
lr = .1
es = 10                                # early-stopping patience: stop after `es` rounds without improvement
n_est = round(500/lr)                  # generous upper bound; early stopping decides actual count
seed = 2

base_params = {
    'n_estimators':n_est
    ,'learning_rate':lr
    ,'verbosity':0                     # library log level: 0=silent, 1=warning, 2=info, 3=debug
    ,'enable_categorical':True         # MUST be True to feed pandas 'category' dtype directly
    ,'early_stopping_rounds':es        # constructor-level (XGBoost ≥1.6); LightGBM uses a fit() callback instead
    ,'random_state':seed
    ,'objective':'reg:squarederror'    # standard L2 regression
    ,'eval_metric':'rmse'              # reported during fit; we compute MAE on OOF later
    ,'device':'cuda'                   # 'cpu' if no GPU; 'cuda:0' to pin a specific device
    ,'reg_lambda':0                    # no L2 on leaf values
    ,'min_child_weight':1              # min sum-of-Hessians per leaf; ≈ min samples for squared error
}
kf_params = {
    'n_splits':3
    ,'n_repeats':1
    ,'random_state':seed
}


## 10. The cross-validation training loop

This is the heart of the notebook. For each of the `n_splits × n_repeats` folds we:

1. Take the train rows in `idx_t` and the validation rows in `idx_v`.
2. Apply the **target power transform** (`y → y^0.5`) to both halves. Sales counts have a heavy
   right tail — a few products sell hundreds of units a day while most sell a handful. A square-root
   transform compresses the tail so the model isn't dominated by huge-volume items. We invert with
   `pred^(1/0.5) = pred^2` after prediction.
3. Fit a fresh `XGBRegressor`. The `eval_set=[(X_v, y_v)]` tells XGBoost where to compute the
   evaluation metric for early stopping. Note that this means the validation set is used for
   model selection (picking `best_iteration`) — for production, hold out a separate inner slice.
4. Predict `test` and `X_v`, undo the power transform, clip to `≥ 0` (sales can't be negative).
5. Store: per-fold test predictions (we'll average them) and OOF predictions (for honest validation).

### Predict the test set every fold

We predict `test` *inside* the fold loop because we want **`n_splits` different test predictions
to average together**. Averaging across folds is a cheap form of bagging that reduces variance,
typically improving leaderboard score. XGBoost will use the model's `best_iteration` for prediction
automatically when early stopping fired.

### `verbose=100*es` in `fit()`

XGBoost will print the eval metric every `verbose` rounds (here every 1000 = 100 × 10). With early
stopping patience of 10 and lr 0.1, training rarely lasts more than a few hundred rounds, so this
effectively prints once or twice per fold — useful as a heartbeat without flooding the output.

### Categorical handling — what's NOT in this cell

You may notice this cell does *not* contain the dtype-cleanup block found in the LightGBM sister
notebook. With `enable_categorical=True`, XGBoost accepts `pandas.CategoricalDtype` columns
directly and uses their integer codes. **However**, you still need to ensure train and test share
the *same* categorical levels — otherwise `'warehouse_3'` could be code 2 in train and code 5 in
test, and the model would treat them as different. The cast `all_data[cat_cols].astype('str').astype('category')`
in the previous cell relies on `all_data` being a single concatenated frame, which guarantees
shared levels. If you ever build train and test as separate frames, recast them via a shared
`CategoricalDtype` (see the LightGBM notebook for that pattern).


In [ ]:
drop_cols = ['date','name','L1_category_name_en']  # date is captured by 'days_since_2020' etc.; name was decomposed; L1 is too coarse
oof_preds = []
test_preds = []
pow_trans = True       # apply y -> y^0.5 power transform
pow_degree = .5

kf = RepeatedKFold(**kf_params)
X, y = deepcopy(X_train), deepcopy(y_train)   # work on copies; original frames stay clean
X[cat_cols] = all_data[cat_cols]              # bring in shared category dtypes (already aligned via all_data)
X.drop(drop_cols, axis=1, inplace=True)
test_copy = deepcopy(test)
test_copy[cat_cols] = all_data[cat_cols]
test_copy.drop(drop_cols, axis=1, inplace=True)

# OOF container: one column per CV repeat. Within a repeat, splits are disjoint, so
# every row gets filled exactly once.
oof_pred_df = pd.DataFrame(index=X.index, columns=[
    'Pred_{0}'.format(i) for i in range(kf_params['n_repeats'])])

for i, (idx_t, idx_v) in enumerate(kf.split(X)):
    print(f"{i} - {len(idx_t)} - {len(idx_v)}")
    X_t, X_v = X.iloc[idx_t], X.iloc[idx_v]
    print(f"X_t cols: {X_t.columns.tolist()}")
    y_t, y_v = y.loc[X_t.index], y.loc[X_v.index]

    # Power transform compresses the heavy right tail of count-like targets.
    if pow_trans:
        y_t, y_v = np.power(y_t, pow_degree), np.power(y_v, pow_degree)

    # Fresh model per fold. Early stopping is configured on the constructor (in base_params).
    xgb = XGBRegressor(**base_params)
    # eval_set is the validation set XGBoost watches for early stopping; verbose=1000 = print every 1000 rounds.
    xgb.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=100*es)

    # Predict test, invert the power transform, clip negatives to zero (sales can't be negative).
    # XGBoost automatically uses best_iteration when early stopping fired.
    model_test_preds = np.power(xgb.predict(test_copy).clip(0), 1/pow_degree) if pow_trans else xgb.predict(test_copy).clip(0)
    test_preds.append(model_test_preds)

    # Predict OOF rows the same way and store in the wide frame.
    model_oof_preds = np.power(xgb.predict(X_v).clip(0), 1/pow_degree) if pow_trans else xgb.predict(X_v).clip(0)
    oof_pred_df.iloc[idx_v, int(i/kf_params['n_splits'])] = model_oof_preds

oof_preds.append(oof_pred_df)
print(len(test_preds))


## 11. Score the out-of-fold predictions

Now we evaluate the model on data it never saw during training (within each fold).

- `oof_pred_df` has one column per repeat → average across columns to get a single OOF prediction per row.
- We compute **weighted MAE** (the competition metric) using the per-product weights from earlier.
- This number is your **honest holdout score**, modulo the time-leakage caveat from the CV section.
  Treat it as an upper bound on test-set quality.

### Comparing to LightGBM

When you've run both notebooks, compare the OOF weighted MAE values. A few possible outcomes:

- **Roughly equal** — typical. The two models extract similar signal from this feature set.
- **One clearly better** — investigate why. Different categorical handling? Different sensitivity
  to a noisy feature? Try matched hyperparameters and look at feature importances.
- **Both close, but their average beats either alone** — also typical. Trees grown level-wise
  (XGBoost) vs leaf-wise (LightGBM), with different tie-breaks and split-finding heuristics, make
  decorrelated errors. Average their test predictions for an easy ensemble lift.


In [ ]:
oof_pred_df = pd.concat(oof_preds, axis=1)
test_pred_df = pd.DataFrame(np.transpose(test_preds), index=test.index)
oof_pred_vals = oof_pred_df.mean(axis=1)
np.round(mean_absolute_error(y_train, oof_pred_vals, sample_weight=X_train_weights), 3)


## 12. Build the submission file

Average test predictions across folds (bagging) and write to CSV. The filename `xgb_submission.csv`
mirrors `lgbm_submission.csv` from the LightGBM notebook — keeping them parallel makes ensembling
later trivial:

```python
lgbm = pd.read_csv('lgbm_submission.csv', index_col=0)['sales_hat']
xgb  = pd.read_csv('xgb_submission.csv',  index_col=0)['sales_hat']
ens  = (lgbm + xgb) / 2     # simple average
# or weighted: 0.6 * lgbm + 0.4 * xgb (tune via OOF)
ens.to_csv('ensemble_submission.csv', header=['sales_hat'])
```


In [ ]:
test_sub = test_pred_df.mean(axis=1)
test_sub.name = 'sales_hat'
test_sub.to_csv('xgb_submission.csv')


## Recap — XGBoost-specific takeaways

The big-picture lessons (cyclical encoding, hierarchical features, target leakage, power transforms,
sample weights, OOF averaging) are the same as in the LightGBM notebook. The XGBoost-specific bits:

1. **`enable_categorical=True` is opt-in.** It was experimental for several versions and is now stable
   (XGBoost ≥1.7). Without it, you're back to manual label/one-hot encoding.
2. **Early stopping is a constructor parameter** (`early_stopping_rounds=es`), not a fit-time callback.
   Cleaner for sklearn pipelines, slightly different from LightGBM's idiom.
3. **GPU is straightforward.** `device='cuda'` works on any CUDA-capable build of XGBoost. Remember
   to flip it back to `'cpu'` when running on a machine without a GPU, or you'll get cryptic CUDA errors.
4. **Objective name is `'reg:squarederror'`**, not `'regression'`. XGBoost uses `family:variant`
   naming (`reg:absoluteerror`, `reg:gamma`, `reg:tweedie`, `binary:logistic`, ...). Worth
   browsing the [objective list](https://xgboost.readthedocs.io/en/stable/parameter.html#learning-task-parameters) —
   `reg:absoluteerror` directly optimizes MAE, which is what the competition scores on, and might
   be worth a try here.
5. **`min_child_weight` is in Hessian space**, not sample space. For L2 loss it's effectively
   "min samples per leaf"; for other objectives the units shift. If you see the model overfit, raise it.
6. **Categorical level alignment** still matters even with `enable_categorical=True`. Ensure train
   and test share the same `CategoricalDtype` (here: by casting on the concatenated `all_data`).
7. **The natural pairing with LightGBM is averaging.** Run both, average test predictions, score
   the OOF average — usually beats either alone.

### Suggested exercises

- Replace `RepeatedKFold` with `TimeSeriesSplit` and recompute the OOF MAE. How much does it change?
- Switch the objective to `'reg:absoluteerror'` (which directly optimizes MAE). Does the OOF MAE drop?
- Add the integer `days_to_next_holiday` and `days_since_last_holiday` (instead of just the binary
  before/after flags). Does the OOF score improve?
- Pass `sample_weight=X_train_weights.iloc[idx_t]` to `xgb.fit()` so training itself optimizes
  the weighted loss. The author left this off; try it both ways.
- Average this notebook's `xgb_submission.csv` with LightGBM's `lgbm_submission.csv`. What's the
  OOF MAE of `(oof_xgb + oof_lgbm) / 2`?
- Tune `learning_rate`, `max_depth`, and `min_child_weight` with Optuna or a coarse grid; see how
  much headroom is left.
